In [ ]:
# 1. Install PaddlePaddle GPU build (CUDA 12.6)
!pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# 2. Install PaddleOCR and upgrade nvidia-nccl-cu12 so PyTorch doesn't crash
!pip install -U paddleocr "nvidia-nccl-cu12>=2.27.3"

# 3. Verify that Paddle can see the GPU
!python -c "import paddle; paddle.utils.run_check()"

In [ ]:
import requests
from paddleocr import PaddleOCR

# 1. Download a sample document image
IMAGE_URL = "https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/general_ocr_002.png"
IMAGE_PATH = "sample_doc.png"

resp = requests.get(IMAGE_URL, timeout=30)
resp.raise_for_status()
with open(IMAGE_PATH, "wb") as f:
    f.write(resp.content)
print(f"Downloaded {len(resp.content) / 1024:.1f} KB -> {IMAGE_PATH}")

# 2. Initialize PaddleOCR for English on the GPU
ocr = PaddleOCR(
    lang="en",
    device="gpu:0",
    use_doc_orientation_classify=False,  # skip page-rotation model (faster)
    use_doc_unwarping=False,             # skip de-warping model (faster)
    use_textline_orientation=False,      # flip to True if lines may be rotated
)

# 3. Run OCR
results = ocr.predict(IMAGE_PATH)

# 4. Print the extracted text with confidence scores
for page in results:
    texts = page["rec_texts"]
    scores = page["rec_scores"]
    print(f"\nDetected {len(texts)} text lines:\n")
    for text, score in zip(texts, scores):
        print(f"[{score:.2f}] {text}")

    # Full text as a single string
    print("\n--- Full text ---")
    print("\n".join(texts))

In [ ]:
!apt-get update
!apt-get install -y poppler-utils
!pip install -q pdf2image opencv-python-headless

In [ ]:
!pip install -q supabase "httpx==0.28.1" "pydantic==2.14.1"

from importlib.metadata import version
for pkg in ["supabase", "httpx", "pydantic", "postgrest", "storage3"]:
    print(f"{pkg:<10} {version(pkg)}")

In [ ]:
# =============================================================================
# CELL 1 -- INSTALLATION  (paste into its own Colab cell, including the "!")
# =============================================================================
# !pip install -q supabase "httpx==0.28.1" "pydantic==2.14.1"
# from importlib.metadata import version
# for pkg in ["supabase", "httpx", "pydantic", "postgrest", "storage3"]:
#     print(f"{pkg:<10} {version(pkg)}")


# =============================================================================
# CELL 2 -- SUPABASE CONNECTION + HEALTH CHECK  (paste into a separate cell)
# =============================================================================
from google.colab import userdata
from supabase import create_client, Client

TEST_CASE_NUMBER = "PA-2026-TEST01"
REQUIRED_BUCKETS = ["raw-medical-docs", "processed-previews"]


def _ok(msg):   print(f"  [PASS] {msg}")
def _fail(msg): print(f"  [FAIL] {msg}")
def _info(msg): print(f"         {msg}")


def _bucket_name(b):
    """Bucket objects differ across SDK versions (object vs dict)."""
    if isinstance(b, dict):
        return b.get("name") or b.get("id")
    return getattr(b, "name", None) or getattr(b, "id", None)


print("=" * 64)
print(" SUPABASE CONNECTION HEALTH CHECK  (Squad 1 - Colab OCR)")
print("=" * 64)

all_checks_passed = True
supabase: Client = None

# ---------------------------------------------------------------- 1. Secrets
print("\n[1/4] Reading secrets from Colab Secrets Manager")
try:
    SUPABASE_URL = userdata.get("SUPABASE_URL")
    SUPABASE_SERVICE_KEY = userdata.get("SUPABASE_SERVICE_KEY")

    if not SUPABASE_URL or not SUPABASE_URL.startswith("https://"):
        raise ValueError("SUPABASE_URL must start with https:// (e.g. https://xxxx.supabase.co)")
    if not SUPABASE_SERVICE_KEY:
        raise ValueError("SUPABASE_SERVICE_KEY is empty")

    _ok("SUPABASE_URL and SUPABASE_SERVICE_KEY loaded")
    _info(f"Project host : {SUPABASE_URL.replace('https://', '').split('/')[0]}")
    _info(f"Service key  : {'*' * 8} ({len(SUPABASE_SERVICE_KEY)} chars, value hidden)")
except userdata.SecretNotFoundError as e:
    all_checks_passed = False
    _fail(f"Secret not found: {e}")
    _info("Fix: click the key icon in Colab's left sidebar and add the secret with the exact name.")
except userdata.NotebookAccessError:
    all_checks_passed = False
    _fail("Notebook does not have access to the secrets.")
    _info("Fix: in the Secrets panel, turn ON 'Notebook access' for both secrets.")
except Exception as e:
    all_checks_passed = False
    _fail(f"{type(e).__name__}: {e}")

# ------------------------------------------------------------- 2. Client init
if all_checks_passed:
    print("\n[2/4] Initialising Supabase client")
    try:
        supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
        _ok("Client object created")
    except Exception as e:
        all_checks_passed = False
        _fail(f"{type(e).__name__}: {e}")
        _info("Fix: re-copy the 'service_role' key (not the 'anon' key) from Project Settings > API.")

# ------------------------------------------------- 3. Database: claims_cases
if supabase is not None:
    print(f"\n[3/4] Database check: claims_cases -> case_number = '{TEST_CASE_NUMBER}'")
    try:
        resp = (
            supabase.table("claims_cases")
            .select("case_id, case_number, case_status")
            .eq("case_number", TEST_CASE_NUMBER)
            .limit(1)
            .execute()
        )
        rows = resp.data or []
        if rows:
            _ok("Test record found")
            _info(f"case_id     : {rows[0]['case_id']}")
            _info(f"case_number : {rows[0]['case_number']}")
            _info(f"case_status : {rows[0]['case_status']}")
        else:
            all_checks_passed = False
            _fail("Connected, but the test record does not exist")
            _info("Fix: re-run the INSERT at the bottom of the Step 2 SQL script.")
    except Exception as e:
        all_checks_passed = False
        _fail(f"{type(e).__name__}: {e}")
        if "does not exist" in str(e) or "PGRST205" in str(e):
            _info("Hint: table not found. Run the Step 2 SQL script in the Supabase SQL Editor.")
        elif "Invalid API key" in str(e) or "JWT" in str(e):
            _info("Hint: the service key is wrong or expired.")

# ------------------------------------------------------- 4. Storage buckets
if supabase is not None:
    print("\n[4/4] Storage check: required buckets")
    try:
        found = {_bucket_name(b) for b in supabase.storage.list_buckets()}
        _info(f"Buckets visible to this key: {sorted(n for n in found if n)}")

        for bucket in REQUIRED_BUCKETS:
            if bucket not in found:
                all_checks_passed = False
                _fail(f"Bucket '{bucket}' NOT found")
                _info("Fix: Storage > New bucket, using the exact name.")
                continue
            try:
                supabase.storage.from_(bucket).list(path="", options={"limit": 1})
                _ok(f"Bucket '{bucket}' exists and is readable")
            except Exception as e:
                all_checks_passed = False
                _fail(f"Bucket '{bucket}' exists but cannot be read: {type(e).__name__}: {e}")
    except Exception as e:
        all_checks_passed = False
        _fail(f"{type(e).__name__}: {e}")

# ------------------------------------------------------------------ Summary
print("\n" + "=" * 64)
if all_checks_passed:
    print(" CLOUD CONNECTION LIVE - Supabase is ready for the OCR pipeline.")
else:
    print(" CONNECTION CHECK FAILED - see the [FAIL] lines above.")
print("=" * 64)

In [ ]:
# ---------------------------------------------------------------
# Colab setup (run once in its own cell, before this module):
#   !apt-get -qq install -y poppler-utils
#   !pip install -q pdf2image opencv-python-headless matplotlib requests
# ---------------------------------------------------------------
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import requests

IMAGE_EXTS = {".png", ".jpg", ".jpeg"}
PDF_DPI = 300


# ---------------------------------------------------------------
# 1. Load a document as a list of page images
# ---------------------------------------------------------------
def load_document_pages(file_path, output_dir="pages"):
    """
    Accepts an image (.png/.jpg/.jpeg) or a .pdf.
    Returns a list of image file paths, one per page.

    - Images are returned as-is (single-element list).
    - PDFs are rendered at 300 DPI, one page at a time (to keep RAM low on
      multi-page scans) and saved as PNGs in `output_dir`.
    """
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    ext = path.suffix.lower()

    if ext in IMAGE_EXTS:
        return [str(path)]

    if ext == ".pdf":
        from pdf2image import convert_from_path, pdfinfo_from_path

        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        n_pages = int(pdfinfo_from_path(str(path))["Pages"])
        page_paths = []
        for page_no in range(1, n_pages + 1):
            # Convert a single page per call so we never hold the whole PDF in memory
            images = convert_from_path(
                str(path), dpi=PDF_DPI, first_page=page_no, last_page=page_no
            )
            page_path = out_dir / f"{path.stem}_page_{page_no:03d}.png"
            images[0].save(page_path, "PNG")
            page_paths.append(str(page_path))
        print(f"Converted {n_pages} PDF page(s) at {PDF_DPI} DPI -> {out_dir}/")
        return page_paths

    raise ValueError(f"Unsupported file type '{ext}'. Use .png, .jpg, .jpeg or .pdf")


# ---------------------------------------------------------------
# 2. Preprocessing helpers
# ---------------------------------------------------------------
def _estimate_skew_angle(gray, max_angle=15.0, work_width=1000):
    """
    Estimate the rotation (in degrees) that makes text lines horizontal,
    using the projection-profile method: when lines are perfectly horizontal,
    the row-wise ink histogram has maximum variance (sharp peaks and gaps).

    Returns the angle to pass to cv2.getRotationMatrix2D (positive = counter-
    clockwise), searched only within [-max_angle, +max_angle].
    The binarisation here is used ONLY to measure the angle; it never touches
    the image that is saved for OCR.
    """
    h, w = gray.shape
    scale = min(1.0, work_width / w)
    small = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else gray

    # Ink = white on black. Otsu is fine for angle estimation only.
    _, bw = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)

    # Ignore a 3% border so dark scanner/phone edges don't dominate the score
    mh, mw = int(bw.shape[0] * 0.03), int(bw.shape[1] * 0.03)
    bw = bw[mh : bw.shape[0] - mh, mw : bw.shape[1] - mw]

    if cv2.countNonZero(bw) < 0.001 * bw.size:  # almost blank page
        return 0.0

    sh, sw = bw.shape
    center = (sw / 2.0, sh / 2.0)

    def score(angle):
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rot = cv2.warpAffine(bw, M, (sw, sh), flags=cv2.INTER_NEAREST, borderValue=0)
        profile = rot.sum(axis=1, dtype=np.float64) / 255.0
        return np.var(profile)

    # Coarse search (1 degree), then fine search (0.1 degree) around the winner
    coarse = np.arange(-max_angle, max_angle + 1e-9, 1.0)
    best = max(coarse, key=score)
    fine = np.arange(max(best - 1.0, -max_angle), min(best + 1.0, max_angle) + 1e-9, 0.1)
    return float(max(fine, key=score))


def _rotate(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, 1.0)
    return cv2.warpAffine(
        img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE
    )


# ---------------------------------------------------------------
# 3. Main preprocessing function
# ---------------------------------------------------------------
def preprocess_medical_image(
    image_path,
    output_path,
    denoise_h=10,          # 7-12 is a good range; higher = smoother but softer strokes
    clahe_clip=2.0,        # 1.5-3.0; higher boosts faded text more (and noise)
    clahe_tile=(8, 8),
    max_skew=15.0,
    min_skew_to_fix=0.3,   # ignore tiny angles to avoid needless resampling blur
    verbose=True,
):
    """
    Grayscale -> NLM denoise -> CLAHE -> deskew (+/- max_skew degrees) -> save.
    No binary thresholding is applied, so PaddleOCR's detector still sees
    natural grey-level strokes. Returns output_path.
    """
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)  # honours EXIF rotation
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")

    # 1. Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. Remove camera grain (non-local means)
    denoised = cv2.fastNlMeansDenoising(
        gray, None, h=denoise_h, templateWindowSize=7, searchWindowSize=21
    )

    # 3. CLAHE: local contrast boost for faded text / stamps without clipping strokes
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_tile)
    enhanced = clahe.apply(denoised)

    # 4. Deskew small rotations
    angle = _estimate_skew_angle(enhanced, max_angle=max_skew)
    if abs(angle) >= min_skew_to_fix:
        enhanced = _rotate(enhanced, angle)
    if verbose:
        print(f"[preprocess] {Path(image_path).name}: skew correction = {angle:+.1f} deg")

    # 5. Save losslessly
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), enhanced)
    return str(output_path)


# ---------------------------------------------------------------
# 4. Demo: preprocess sample_doc.png, show before/after, run PaddleOCR
# ---------------------------------------------------------------
def show_side_by_side(original_path, cleaned_path):
    orig = cv2.cvtColor(cv2.imread(str(original_path)), cv2.COLOR_BGR2RGB)
    clean = cv2.imread(str(cleaned_path), cv2.IMREAD_GRAYSCALE)

    fig, axes = plt.subplots(1, 2, figsize=(14, 8))
    axes[0].imshow(orig)
    axes[0].set_title("Original")
    axes[1].imshow(clean, cmap="gray")
    axes[1].set_title("Cleaned (denoise + CLAHE + deskew)")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def run_demo(sample_path="sample_doc.png", cleaned_path="sample_doc_clean.png"):
    from paddleocr import PaddleOCR

    # Download the sample if it isn't already in the Colab filesystem
    if not Path(sample_path).exists():
        url = "https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/general_ocr_002.png"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        Path(sample_path).write_bytes(r.content)

    # Works the same for images and PDFs: one path per page
    pages = load_document_pages(sample_path)

    cleaned_pages = []
    for i, page in enumerate(pages):
        out = cleaned_path if len(pages) == 1 else f"{Path(cleaned_path).stem}_{i + 1:03d}.png"
        cleaned_pages.append(preprocess_medical_image(page, out))

    show_side_by_side(pages[0], cleaned_pages[0])

    ocr = PaddleOCR(
        lang="en",
        device="gpu:0",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )

    for page_no, cleaned in enumerate(cleaned_pages, start=1):
        for res in ocr.predict(cleaned):
            texts, scores = res["rec_texts"], res["rec_scores"]
            avg = sum(scores) / len(scores) if scores else 0.0
            print(f"\n=== Page {page_no}: {len(texts)} lines, mean confidence {avg:.2f} ===")
            for t, s in zip(texts, scores):
                print(f"[{s:.2f}] {t}")


if __name__ == "__main__":
    run_demo()

In [ ]:
# ---------------------------------------------------------------
# Cell 5: Socratic Defense Q1 fallback architecture
#   quality gate -> stamp removal + preprocessing -> PaddleOCR
#   -> calibrated 3-tier routing -> structured JSON for PII Redaction
#
# Needs (from earlier cells): paddleocr, pdf2image/poppler, load_document_pages
# ---------------------------------------------------------------
import json
import re
import unicodedata
from datetime import date
from pathlib import Path

import cv2
import numpy as np
import requests

# Reuse load_document_pages from the earlier cell; fall back to a compact copy
try:
    load_document_pages  # noqa: B018
except NameError:
    def load_document_pages(file_path, output_dir="pages"):
        p = Path(file_path)
        if p.suffix.lower() in {".png", ".jpg", ".jpeg"}:
            return [str(p)]
        if p.suffix.lower() == ".pdf":
            from pdf2image import convert_from_path
            Path(output_dir).mkdir(parents=True, exist_ok=True)
            paths = []
            for i, im in enumerate(convert_from_path(str(p), dpi=300), start=1):
                out = Path(output_dir) / f"{p.stem}_page_{i:03d}.png"
                im.save(out, "PNG")
                paths.append(str(out))
            return paths
        raise ValueError(f"Unsupported file type: {p.suffix}")


# ===============================================================
# 1. Image quality gate
# ===============================================================
def check_image_quality(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return {"quality_pass": False, "sharpness": 0.0, "contrast": 0.0, "brightness": 0.0}

    blurred = cv2.GaussianBlur(img, (3, 3), 0)
    lap = cv2.Laplacian(blurred, cv2.CV_64F)

    # Measure sharpness on the active ink/edge regions (top 10% gradient pixels)
    # so large blank white margins on clean PDFs don't dilute the score
    abs_lap = np.abs(lap)
    p90 = np.percentile(abs_lap, 90)
    edge_pixels = lap[abs_lap >= p90]
    sharpness = float(edge_pixels.var()) if edge_pixels.size > 0 else float(lap.var())

    contrast = float(img.std())
    brightness = float(img.mean())

    # Clean digital PDFs/PNGs have white backgrounds (brightness up to 252, std >= 8)
    quality_pass = (sharpness >= 25.0) and (contrast >= 8.0) and (35.0 <= brightness <= 252.0)
    return {
        "quality_pass": bool(quality_pass),
        "sharpness": round(sharpness, 3),
        "contrast": round(contrast, 3),
        "brightness": round(brightness, 3),
    }


# ===============================================================
# 2. Red-stamp removal + preprocessing
# ===============================================================
def _estimate_skew_angle(gray, max_angle=15.0, work_width=1000):
    """Projection-profile skew search. Returns the angle for
    cv2.getRotationMatrix2D (positive = counter-clockwise). The Otsu
    binarisation is used only to measure the angle, never saved."""
    h, w = gray.shape
    scale = min(1.0, work_width / w)
    small = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else gray
    _, bw = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    mh, mw = int(bw.shape[0] * 0.03), int(bw.shape[1] * 0.03)
    bw = bw[mh: bw.shape[0] - mh, mw: bw.shape[1] - mw]
    if cv2.countNonZero(bw) < 0.001 * bw.size:
        return 0.0

    sh, sw = bw.shape
    center = (sw / 2.0, sh / 2.0)

    def score(angle):
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rot = cv2.warpAffine(bw, M, (sw, sh), flags=cv2.INTER_NEAREST, borderValue=0)
        return np.var(rot.sum(axis=1, dtype=np.float64) / 255.0)

    best = max(np.arange(-max_angle, max_angle + 1e-9, 1.0), key=score)
    fine = np.arange(max(best - 1.0, -max_angle), min(best + 1.0, max_angle) + 1e-9, 0.1)
    return float(max(fine, key=score))


def _rotate(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


def _remove_red_stamp(img_bgr, max_coverage=0.35, verbose=True):
    """Detect red/pink ink in HSV and paint it over with the local median
    background colour. Returns the cleaned BGR image."""
    h, w = img_bgr.shape[:2]
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (0, 40, 40), (15, 255, 255)) | cv2.inRange(
        hsv, (165, 40, 40), (180, 255, 255)
    )
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=2)

    coverage = cv2.countNonZero(mask) / float(mask.size)
    if coverage == 0:
        return img_bgr
    if coverage > max_coverage:
        # A mask this large is a warm/pink colour cast, not a stamp: leave it alone
        if verbose:
            print(f"[stamp] mask covers {coverage:.0%} of the page -> treated as colour cast, skipped")
        return img_bgr

    # Global paper colour from non-stamp pixels (subsampled for speed)
    sub = img_bgr[::6, ::6]
    sub_mask = mask[::6, ::6] == 0
    paper = np.median(sub[sub_mask], axis=0) if sub_mask.any() else np.array([255, 255, 255])

    # Local median of the image with stamp pixels pre-filled by the paper colour
    fill = img_bgr.copy()
    fill[mask > 0] = paper.astype(np.uint8)
    scale = 0.25 if max(h, w) > 1600 else 1.0
    small = cv2.resize(fill, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else fill
    k = max(15, (int(max(small.shape[:2]) / 60) | 1))  # odd kernel
    bg_small = cv2.medianBlur(small, k)
    bg = cv2.resize(bg_small, (w, h), interpolation=cv2.INTER_LINEAR) if scale < 1 else bg_small

    out = img_bgr.copy()
    out[mask > 0] = bg[mask > 0]
    if verbose:
        print(f"[stamp] removed red/pink ink from {coverage:.1%} of pixels")
    return out


def remove_red_stamp_and_preprocess(image_path, output_path, verbose=True):
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")

    # 1. Stamp removal
    img = _remove_red_stamp(img, verbose=verbose)

    # 2. Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 3. Uneven-illumination (phone shadow) normalisation by morphological division.
    #    Background = closing with a large ellipse (erases text, keeps lighting),
    #    estimated on a downscaled copy because lighting is low-frequency.
    h, w = gray.shape
    scale = 0.25 if max(h, w) > 1200 else 1.0
    small = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else gray
    k = max(15, min(51, int(min(small.shape) / 12) | 1))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    bg_small = cv2.GaussianBlur(cv2.morphologyEx(small, cv2.MORPH_CLOSE, kernel), (0, 0), k / 4.0)
    bg = cv2.resize(bg_small, (w, h), interpolation=cv2.INTER_LINEAR) if scale < 1 else bg_small
    normalized = cv2.divide(gray, np.maximum(bg, 1), scale=255)

    # 4. Denoise camera grain
    denoised = cv2.fastNlMeansDenoising(normalized, None, h=10, templateWindowSize=7, searchWindowSize=21)

    # 5. CLAHE
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(denoised)

    # 6. Deskew (-15..+15 degrees)
    angle = _estimate_skew_angle(enhanced, max_angle=15.0)
    if abs(angle) >= 0.3:
        enhanced = _rotate(enhanced, angle)
    if verbose:
        print(f"[preprocess] {Path(image_path).name}: skew correction {angle:+.1f} deg")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), enhanced)
    return str(output_path)


# ===============================================================
# 3. Confidence calibration + 3-tier routing
# ===============================================================
# Matches "500mg", "Tab.", "TDS", "Discharge summary" but not "capital" or "html"
_CLINICAL_RE = re.compile(
    r"(?<![a-z])(?:diagnos\w*|procedure\w*|rx|tabs?|tablets?|inj|injections?|caps?|capsules?"
    r"|mg|ml|tds|bd|od|admit\w*|admission|discharge\w*|surg\w*)(?![a-z])",
    re.IGNORECASE,
)
_DATE_RE = re.compile(r"(?<!\d)(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})(?!\d)")


def _has_valid_date(text):
    """Day-first (Indian) dates; 2- or 4-digit years; rejects 31/02/2024 etc."""
    for d, m, y in _DATE_RE.findall(text):
        if len(y) not in (2, 4):
            continue
        year = int(y) + 2000 if len(y) == 2 else int(y)
        try:
            if 1900 <= year <= 2100:
                date(year, int(m), int(d))
                return True
        except ValueError:
            continue
    return False


def _is_noisy(text):
    """True for empty/single-character strings or strings that are mostly punctuation."""
    s = "".join(text.split())
    if len(s) <= 1:
        return True
    meaningful = sum(unicodedata.category(c)[0] in "LNM" for c in s)  # letters, digits, marks
    return meaningful / len(s) < 0.5


_ROUTING_THRESHOLDS = {  # (auto_at_or_above, confirm_at_or_above)
    True: (0.90, 0.60),    # critical clinical fields
    False: (0.80, 0.50),   # everything else
}


def calibrate_and_route(text, raw_score):
    raw_score = float(raw_score)
    text = text or ""

    has_date_pattern = bool(_DATE_RE.search(text))
    has_clinical_term = bool(_CLINICAL_RE.search(text))
    is_critical = has_clinical_term or has_date_pattern

    # Lightweight Platt-style adjustment (heuristic shift, not a trained sigmoid)
    calibrated = raw_score
    if has_clinical_term or _has_valid_date(text):
        calibrated = max(calibrated, min(0.99, calibrated + 0.03))
    if _is_noisy(text):
        calibrated -= 0.08
    calibrated = float(min(1.0, max(0.0, calibrated)))

    auto_t, confirm_t = _ROUTING_THRESHOLDS[is_critical]
    if calibrated >= auto_t:
        routing = "AUTO"
    elif calibrated >= confirm_t:
        routing = "SUGGEST_CONFIRM"
    else:
        routing = "HUMAN_REVIEW"

    # Only AUTO text may flow downstream; everything else waits for a human
    if routing == "AUTO":
        verified_status, blocked, downstream_value = "AUTO_ACCEPTED", False, text
    elif routing == "SUGGEST_CONFIRM":
        verified_status, blocked, downstream_value = "PENDING_CONFIRMATION", True, None
    else:
        verified_status, blocked, downstream_value = "UNVERIFIED", True, None

    return {
        "is_critical": is_critical,
        "calibrated_score": round(calibrated, 4),
        "routing_decision": routing,
        "verified_status": verified_status,
        "blocked_downstream": blocked,
        "downstream_value": downstream_value,
    }


# ===============================================================
# 4. End-to-end: document -> JSON contract
# ===============================================================
_OCR_ENGINE = None


def get_ocr_engine():
    """Dedicated engine with doc orientation/unwarping OFF so that dt_polys are
    in the same pixel space as the preprocessed page image we crop from."""
    global _OCR_ENGINE
    if _OCR_ENGINE is None:
        from paddleocr import PaddleOCR
        _OCR_ENGINE = PaddleOCR(
            lang="en",
            device="gpu:0",
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
        )
    return _OCR_ENGINE


def _get(res, key, default=None):
    try:
        return res[key]
    except (KeyError, TypeError):
        return default


def _poly_to_list(poly):
    return [[int(round(float(x))), int(round(float(y)))] for x, y in np.asarray(poly).reshape(-1, 2)]


def _save_snippet(gray_img, poly_pts, path, pad=6):
    pts = np.asarray(poly_pts)
    h, w = gray_img.shape[:2]
    x0, y0 = max(int(pts[:, 0].min()) - pad, 0), max(int(pts[:, 1].min()) - pad, 0)
    x1, y1 = min(int(pts[:, 0].max()) + pad, w), min(int(pts[:, 1].max()) + pad, h)
    if x1 <= x0 or y1 <= y0:
        return None
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), gray_img[y0:y1, x0:x1])
    return str(path)


def process_document_to_json(file_path, output_dir="ocr_output", ocr_engine=None):
    engine = ocr_engine or get_ocr_engine()
    out_dir = Path(output_dir)
    snippet_dir = out_dir / "snippets"
    clean_dir = out_dir / "preprocessed"
    for d in (snippet_dir, clean_dir):
        d.mkdir(parents=True, exist_ok=True)

    doc_id = re.sub(r"[^A-Za-z0-9_-]+", "_", Path(file_path).stem)
    page_paths = load_document_pages(file_path)

    pages_out, safe_lines, raw_lines = [], [], []
    summary = {"AUTO": 0, "SUGGEST_CONFIRM": 0, "HUMAN_REVIEW": 0}
    held_by_rfi = 0

    for page_no, page_path in enumerate(page_paths, start=1):
        quality = check_image_quality(page_path)                       # on the original page
        clean_path = str(clean_dir / f"{doc_id}_p{page_no}_clean.png")
        remove_red_stamp_and_preprocess(page_path, clean_path)
        clean_gray = cv2.imread(clean_path, cv2.IMREAD_GRAYSCALE)

        blocks = []
        for res in engine.predict(clean_path):
            texts = list(_get(res, "rec_texts", []))
            scores = list(_get(res, "rec_scores", []))
            polys = _get(res, "dt_polys", None)
            if polys is None or len(polys) != len(texts):     # fall back to rec_polys
                polys = _get(res, "rec_polys", [])

            for text, score, poly in zip(texts, scores, polys):
                if not str(text).strip():
                    continue                                   # empty detections carry no information
                block_id = len(blocks) + 1
                routed = calibrate_and_route(str(text), score)

                snippet_path = None
                if routed["routing_decision"] in ("HUMAN_REVIEW", "SUGGEST_CONFIRM"):
                    snippet_path = _save_snippet(
                        clean_gray, poly, snippet_dir / f"{doc_id}_p{page_no}_b{block_id}.png"
                    )

                blocks.append({
                    "block_id": block_id,
                    "raw_text": str(text),
                    "downstream_value": routed["downstream_value"],
                    "verified_status": routed["verified_status"],
                    "is_critical_field": routed["is_critical"],
                    "raw_confidence": round(float(score), 4),
                    "calibrated_confidence": routed["calibrated_score"],
                    "routing_decision": routed["routing_decision"],
                    "blocked_downstream": routed["blocked_downstream"],
                    "bounding_box": _poly_to_list(poly),
                    "cropped_snippet_path": snippet_path,
                })

        # ---- 40% page-level RFI rule ----
        total = len(blocks)
        n_human = sum(b["routing_decision"] == "HUMAN_REVIEW" for b in blocks)
        ratio = (n_human / total) if total else 1.0
        rfi_reasons = []
        if not quality["quality_pass"]:
            rfi_reasons.append("IMAGE_QUALITY_FAIL")
        if total == 0:
            rfi_reasons.append("NO_TEXT_DETECTED")
        elif ratio > 0.40:
            rfi_reasons.append(f"HUMAN_REVIEW_RATIO_{ratio:.2f}_EXCEEDS_0.40")
        page_action = "TRIGGER_PAGE_RFI_RESCAN" if rfi_reasons else "PARTIAL_OR_FULL_EXTRACTION"

        # A page that needs a rescan must not feed downstream, even its AUTO blocks
        if page_action == "TRIGGER_PAGE_RFI_RESCAN":
            for b in blocks:
                if b["routing_decision"] == "AUTO":
                    b["downstream_value"] = None
                    b["blocked_downstream"] = True
                    b["verified_status"] = "HELD_PAGE_RESCAN"
                    held_by_rfi += 1

        for b in blocks:
            summary[b["routing_decision"]] += 1
            raw_lines.append(b["raw_text"])
            if b["routing_decision"] == "AUTO" and not b["blocked_downstream"]:
                safe_lines.append(b["raw_text"])

        pages_out.append({
            "page_number": page_no,
            "quality_metrics": quality,
            "page_action": page_action,
            "rfi_reasons": rfi_reasons,
            "preprocessed_image_path": clean_path,
            "text_blocks": blocks,
        })

    # ---- document status ----
    rfi_pages = sum(p["page_action"] == "TRIGGER_PAGE_RFI_RESCAN" for p in pages_out)
    if rfi_pages and rfi_pages == len(pages_out):
        status = "RFI_RESCAN_REQUIRED"
    elif rfi_pages:
        status = "PARTIAL_RFI_RESCAN"
    elif summary["HUMAN_REVIEW"] or summary["SUGGEST_CONFIRM"]:
        status = "PENDING_HUMAN_REVIEW"
    else:
        status = "AUTO_COMPLETE"

    result = {
        "document_id": doc_id,
        "source_file": str(file_path),
        "total_pages": len(pages_out),
        "document_status": status,
        "safe_downstream_text": "\n".join(safe_lines),
        "full_raw_audit_text": "\n".join(raw_lines),
        "audit_summary": {**summary, "auto_blocks_held_by_page_rfi": held_by_rfi},
        "pages": pages_out,
    }

    json_path = out_dir / f"{doc_id}_ocr.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"Saved JSON contract -> {json_path}")
    return result


# ===============================================================
# 5. Test on sample_doc.png
# ===============================================================
if __name__ == "__main__":
    sample = "sample_doc.png"
    if not Path(sample).exists():
        url = "https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/general_ocr_002.png"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        Path(sample).write_bytes(r.content)

    result = process_document_to_json(sample)

    summary = {k: result[k] for k in
               ("document_id", "source_file", "total_pages", "document_status", "audit_summary")}
    summary["pages"] = [
        {"page_number": p["page_number"], "quality_metrics": p["quality_metrics"],
         "page_action": p["page_action"], "rfi_reasons": p["rfi_reasons"],
         "n_blocks": len(p["text_blocks"])}
        for p in result["pages"]
    ]
    summary["safe_downstream_text_chars"] = len(result["safe_downstream_text"])
    print("\n=== JSON SUMMARY ===")
    print(json.dumps(summary, indent=2, ensure_ascii=False))

    print("\n=== FIRST 3 TEXT BLOCKS (page 1) ===")
    print(json.dumps(result["pages"][0]["text_blocks"][:3], indent=2, ensure_ascii=False))

In [ ]:
# ---------------------------------------------------------------
# Cell 6 (Supabase edition): Hybrid Case-Bundle batch pipeline
#
#   claims_cases   (pipeline_stage = 'PENDING_OCR')        -> one case per row
#   claim_documents (case_id = case)                       -> one row per uploaded file
#   raw-medical-docs/{case_id_uuid}/{doc_id}_{file_name}   -> source files (downloaded for OCR)
#   processed-previews/{case_number}/snippets/*.png        -> SUGGEST_CONFIRM / HUMAN_REVIEW crops
#   processed-previews/{case_number}/preprocessed/*.png    -> page images the bounding boxes refer to
#
#   claim_documents.raw_ocr_json            <- one document JSON per file (the cache)
#   claims_cases.master_ocr_bundle_json     <- {case_id}_master_bundle.json (Section 6.2 schema)
#   claims_cases.case_audit_summary         <- bundle["case_audit_summary"]
#   claims_cases.pipeline_stage             <- PENDING_PII | RFI_RESCAN_REQUIRED
#
# Run Cells 1-5 first: this cell calls process_document_to_json() / get_ocr_engine()
# from Cell 5 and uses the `supabase` client from the connection cell.
# ---------------------------------------------------------------
import json
import posixpath
import re
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

# ---------------------------------------------------------------
# 0. Configuration
# ---------------------------------------------------------------
RAW_BUCKET = "raw-medical-docs"
PREVIEW_BUCKET = "processed-previews"
TMP_ROOT = Path("/tmp/ocr_case_runner")

SUPPORTED_EXTS = {".pdf", ".png", ".jpg", ".jpeg"}

# claims_cases.pipeline_stage values
STAGE_PENDING_OCR = "PENDING_OCR"
STAGE_PENDING_PII = "PENDING_PII"
STAGE_RFI_RESCAN = "RFI_RESCAN_REQUIRED"

# claim_documents.processing_status values (same ones the earlier cloud worker used)
DOC_PENDING_OCR = "PENDING_OCR"
DOC_PENDING_PII = "PENDING_PII"
DOC_RFI = "RFI_BLURRY_SCAN"
DOC_FAILED = "FAILED"
# A row in one of these states is never trusted as a finished OCR result, even if
# raw_ocr_json happens to be populated (e.g. left over from an earlier upload).
_NON_CACHEABLE_STATUSES = {DOC_PENDING_OCR, "OCR_PROCESSING", DOC_FAILED}


# ---------------------------------------------------------------
# 1. Small helpers
# ---------------------------------------------------------------
def _g(obj, key, default=None):
    """Read a field from a dict OR an SDK object (storage3 differs across versions)."""
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def _safe_name(name):
    """Storage-key / folder-safe name (letters, digits, _ and -)."""
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(name)).strip("_") or "UNNAMED"


def _safe_filename(name):
    """Local file name: keep the extension, drop anything odd."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(name)) or "UNNAMED"


def _doc_id_from_name(name):
    # Must mirror the document_id logic in process_document_to_json (Cell 5)
    return re.sub(r"[^A-Za-z0-9_-]+", "_", Path(name).stem)


def _now_iso():
    return datetime.now(timezone.utc).isoformat()


def _json_safe(obj):
    """Guarantee plain JSON (no numpy types, no Paths) before it reaches Postgres."""
    return json.loads(json.dumps(obj, ensure_ascii=False, default=str))


def _annotate(doc, **extra):
    """Return a copy of `doc` with extra keys placed right after 'source_file'."""
    out = {}
    for k, v in doc.items():
        if k in extra:
            continue
        out[k] = v
        if k == "source_file":
            out.update(extra)
    if "source_file" not in doc:
        out.update(extra)
    return out


def _clean_error(e):
    return " ".join(f"{type(e).__name__}: {e}".split())[:300]   # one line, no poppler spam


def _upload_png(bucket, key, data, retries=3):
    """Upload with upsert + small retry, so a transient network blip does not fail a document."""
    for attempt in range(1, retries + 1):
        try:
            supabase.storage.from_(bucket).upload(
                path=key,
                file=data,
                file_options={"content-type": "image/png", "upsert": "true"},
            )
            return
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * attempt)


# ---------------------------------------------------------------
# 2. Weighted regex document classifier
# ---------------------------------------------------------------
# {category: {regex: weight}}. Each pattern counts once per document, so a long
# page full of "mg" cannot outvote a clear "Discharge Summary" heading.
_CATEGORY_LEXICON = {
    "DISCHARGE_SUMMARY": {
        r"discharge\s+(?:summary|card|advice|medications?|instructions?)": 5,
        r"date\s+of\s+(?:admission|discharge)": 4,
        r"condition\s+at\s+discharge": 3,
        r"hospital\s+course|course\s+in\s+(?:the\s+)?hospital": 3,
        r"final\s+diagnosis": 2,
        r"\b(?:doa|dod|ipd)\b": 2,
        r"\bdischarged?\b": 1,
        r"\badmi(?:tted|ssion)\b": 1,
        r"follow[\s-]?up": 1,
    },
    "LAB_REPORT": {
        r"reference\s+(?:range|interval|value)s?|\bnormal\s+range\b|bio\.?\s*ref": 4,
        r"laborator(?:y|ies)|pathology|h(?:a)?ematology|biochemistry|microbiology": 3,
        r"g/dl|mg/dl|mmol/l|iu/l|\bu/l\b|/cumm|\bfl\b|\bpg\b": 2,
        r"\b(?:specimen|sample\s+collected|collected\s+on)\b": 2,
        r"h(?:a)?emoglobin|\bhb\b|\bwbc\b|\brbc\b|platelet|\bcbc\b|creatinine|bilirubin": 1,
        r"\b(?:sgpt|sgot|alt|ast|hba1c|tsh|lft|kft|urea|serum|glucose|cholesterol|lipid)\b": 1,
    },
    "PRESCRIPTION": {
        r"\brx\b|\u211e": 4,
        r"prescription": 3,
        r"(?:after|before)\s+(?:food|meals?)": 2,
        r"\bx\s*\d+\s*(?:days?|weeks?)\b|\bfor\s+\d+\s*days?\b": 2,
        r"\b(?:tabs?|tablets?|caps?|capsules?|syp|syrup|inj|injections?|oint|drops)\b": 1,
        r"\b(?:tds|bd|od|qid|bid|tid|sos|hs|stat)\b": 1,
        r"\b\d+\s*(?:mg|ml|mcg)\b": 1,
    },
    "CLINICAL_NOTE": {
        r"chief\s+complaints?|presenting\s+complaints?|\bc/o\b": 3,
        r"\bh/o\b|history\s+of|\bhpi\b|past\s+history": 2,
        r"\bo/e\b|on\s+examination|general\s+examination": 2,
        r"assessment|impression|progress\s+note|\bopd\b|consultation": 2,
        r"\b(?:bp|pulse|spo2|temp(?:erature)?|rr)\b\s*[:\-]?\s*\d": 1,
    },
}
_CATEGORY_RE = {
    cat: [(re.compile(p, re.IGNORECASE), w) for p, w in patterns.items()]
    for cat, patterns in _CATEGORY_LEXICON.items()
}
_MIN_CATEGORY_SCORE = 3      # below this the document falls back to CLINICAL_NOTE


def guess_document_category(text):
    """Returns 'DISCHARGE_SUMMARY' | 'LAB_REPORT' | 'PRESCRIPTION' | 'CLINICAL_NOTE'.
    Ties go to the earlier category in that order; weak evidence -> CLINICAL_NOTE."""
    text = text or ""
    scores = {
        cat: sum(w for rx, w in rules if rx.search(text))
        for cat, rules in _CATEGORY_RE.items()
    }
    best_cat, best = "CLINICAL_NOTE", 0
    for cat in ("DISCHARGE_SUMMARY", "LAB_REPORT", "PRESCRIPTION", "CLINICAL_NOTE"):
        if scores[cat] > best:
            best_cat, best = cat, scores[cat]
    return best_cat if best >= _MIN_CATEGORY_SCORE else "CLINICAL_NOTE"


# ---------------------------------------------------------------
# 3. Case discovery (Supabase)
# ---------------------------------------------------------------
def fetch_cases_to_process(case_numbers=None):
    """
    Returns a list of claims_cases rows (case_id, case_number, pipeline_stage).

    Default: every case whose pipeline_stage is PENDING_OCR, PLUS any case that holds a
    claim_documents row still in PENDING_OCR (e.g. a hospital re-uploaded one file for a
    case that was already waiting in RFI_RESCAN_REQUIRED).
    With case_numbers=[...]: exactly those cases, whatever their stage (manual re-run).
    """
    cols = "case_id, case_number, pipeline_stage"
    if case_numbers:
        rows = (supabase.table("claims_cases").select(cols)
                .in_("case_number", list(case_numbers)).execute().data or [])
    else:
        rows = (supabase.table("claims_cases").select(cols)
                .eq("pipeline_stage", STAGE_PENDING_OCR).execute().data or [])
        have = {r["case_id"] for r in rows}
        pending_docs = (supabase.table("claim_documents").select("case_id")
                        .eq("processing_status", DOC_PENDING_OCR).execute().data or [])
        extra = sorted({r["case_id"] for r in pending_docs} - have)
        if extra:
            rows += (supabase.table("claims_cases").select(cols)
                     .in_("case_id", extra).execute().data or [])
    return sorted(rows, key=lambda r: str(r["case_number"]))


def fetch_case_documents(case_id):
    """All claim_documents of a case (cached + new), in a stable order."""
    return (supabase.table("claim_documents")
            .select("doc_id, case_id, file_name, raw_storage_path, processing_status, "
                    "raw_ocr_json, source_size_bytes")
            .eq("case_id", case_id)
            .order("file_name")
            .execute().data or [])


# ---------------------------------------------------------------
# 4. Remote file size (for smart caching)
# ---------------------------------------------------------------
def _remote_size(raw_path, size_index):
    """Size in bytes of an object in raw-medical-docs, from the folder listing (one list
    call per folder, memoised in size_index). Returns None if the listing has no size."""
    folder, name = posixpath.split(raw_path)
    if folder not in size_index:
        sizes, offset = {}, 0
        while True:
            items = supabase.storage.from_(RAW_BUCKET).list(
                path=folder, options={"limit": 1000, "offset": offset}
            ) or []
            for it in items:
                meta = _g(it, "metadata") or {}
                size = _g(meta, "size") or _g(meta, "contentLength")
                if _g(it, "name") and size is not None:
                    sizes[_g(it, "name")] = int(size)
            if len(items) < 1000:
                break
            offset += 1000
        size_index[folder] = sizes
    return size_index[folder].get(name)


def _download_raw(raw_path):
    return supabase.storage.from_(RAW_BUCKET).download(raw_path)


# ---------------------------------------------------------------
# 5. Per-document processing (smart caching + GPU OCR)
# ---------------------------------------------------------------
def _is_cache_candidate(row):
    j = row.get("raw_ocr_json")
    return (
        isinstance(j, dict)
        and "document_id" in j and "pages" in j
        and row.get("processing_status") not in _NON_CACHEABLE_STATUSES
        and row.get("source_size_bytes") is not None
    )


def _doc_db_status(doc):
    """claim_documents.processing_status for a freshly OCR'd document."""
    if doc.get("document_status") in ("RFI_RESCAN_REQUIRED", "PARTIAL_RFI_RESCAN"):
        return DOC_RFI
    return DOC_PENDING_PII


def _rfi_reason_text(doc):
    parts = []
    for p in doc.get("pages", []):
        if p.get("page_action") == "TRIGGER_PAGE_RFI_RESCAN":
            parts.append(f"page {p['page_number']}: {', '.join(p.get('rfi_reasons', []))}")
    if not parts:
        return None
    return "Automated RFI - please upload a clearer scan. " + "; ".join(parts)


def _save_document_row(row, doc, size_bytes, preview_paths):
    blocks = [b for p in doc["pages"] for b in p["text_blocks"]]
    scores = [b["raw_confidence"] for b in blocks]
    status = _doc_db_status(doc)
    update = {
        "raw_ocr_json": doc,
        "source_size_bytes": size_bytes,
        "processing_status": status,
        "preview_storage_paths": preview_paths,
        "avg_ocr_confidence": round(sum(scores) / len(scores), 4) if scores else 0.0,
        "low_conf_line_count": sum(b["routing_decision"] == "HUMAN_REVIEW" for b in blocks),
        "is_confidence_acceptable": status == DOC_PENDING_PII,
        "rfi_reason": _rfi_reason_text(doc) if status == DOC_RFI else None,
        "ocr_completed_at": _now_iso(),
    }
    supabase.table("claim_documents").update(update).eq("doc_id", row["doc_id"]).execute()
    return status


def _process_one_document(row, case_number, size_index, force=False):
    """
    Returns a dict: status ('CACHED' | 'SUCCESS'), doc, gpu_time, snippets_uploaded.
    Raises on failure (the caller records FAILED and carries on with the next file).
    """
    raw_path = row.get("raw_storage_path")
    if not raw_path or raw_path == "PENDING_UPLOAD":
        raise ValueError("file has not been uploaded to raw-medical-docs yet")

    safe_case = _safe_name(case_number)
    data = None

    # ---- Smart caching: reuse raw_ocr_json when the file is unchanged -------------
    if not force and _is_cache_candidate(row):
        remote_size = _remote_size(raw_path, size_index)
        if remote_size is None:                       # listing gave no size: measure it ourselves
            data = _download_raw(raw_path)
            remote_size = len(data)
        if remote_size == row["source_size_bytes"]:
            cached = row["raw_ocr_json"]
            cat = guess_document_category(cached.get("full_raw_audit_text", ""))
            if cached.get("detected_doc_category") != cat:        # keep tag in sync with lexicon
                cached = _annotate(cached, detected_doc_category=cat)
                supabase.table("claim_documents").update(
                    {"raw_ocr_json": cached}
                ).eq("doc_id", row["doc_id"]).execute()
            return {"status": "CACHED", "doc": cached, "gpu_time": 0.0, "snippets_uploaded": 0}

    # ---- GPU OCR on a new / modified file -----------------------------------------
    if data is None:
        data = _download_raw(raw_path)
    size_bytes = len(data)

    work_dir = TMP_ROOT / str(row["doc_id"])
    local_file = work_dir / _safe_filename(row["file_name"])
    try:
        work_dir.mkdir(parents=True, exist_ok=True)
        local_file.write_bytes(data)

        engine = get_ocr_engine()          # load model before the timer so load time is not billed to the doc
        t0 = time.perf_counter()
        result = process_document_to_json(
            str(local_file), output_dir=str(work_dir / "ocr_output"), ocr_engine=engine
        )
        gpu_time = time.perf_counter() - t0

        # Upload page images + review snippets; swap local /tmp paths for storage paths
        preview_paths, n_snippets = [], 0
        for page in result["pages"]:
            pre = page.get("preprocessed_image_path")
            if pre and Path(pre).exists():
                key = f"{safe_case}/preprocessed/{Path(pre).name}"
                _upload_png(PREVIEW_BUCKET, key, Path(pre).read_bytes())
                page["preprocessed_image_path"] = key
                preview_paths.append(key)

            for b in page["text_blocks"]:
                snip = b.get("cropped_snippet_path")
                if not snip:
                    continue                                   # AUTO blocks carry no snippet
                if Path(snip).exists():
                    key = f"{safe_case}/snippets/{Path(snip).name}"
                    _upload_png(PREVIEW_BUCKET, key, Path(snip).read_bytes())
                    b["cropped_snippet_path"] = key
                    n_snippets += 1
                else:
                    b["cropped_snippet_path"] = None

        result["source_file"] = raw_path
        doc = _annotate(
            result,
            source_size_bytes=size_bytes,
            detected_doc_category=guess_document_category(result.get("full_raw_audit_text", "")),
        )
        doc = _json_safe(doc)

        _save_document_row(row, doc, size_bytes, preview_paths)
        return {"status": "SUCCESS", "doc": doc, "gpu_time": gpu_time, "snippets_uploaded": n_snippets}
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)
        # load_document_pages() renders PDF pages into ./pages/ - remove them too
        for leftover in Path("pages").glob(f"{local_file.stem}_page_*.png"):
            leftover.unlink(missing_ok=True)


# ---------------------------------------------------------------
# 6. Case master bundle  (Section 6.2 schema)
# ---------------------------------------------------------------
def _build_case_bundle(case_id, documents, failed_docs):
    """`documents` is a list of (file_name, doc_dict) pairs; the bundle itself only stores doc_dicts."""
    audit = {"AUTO": 0, "SUGGEST_CONFIRM": 0, "HUMAN_REVIEW": 0, "auto_blocks_held_by_page_rfi": 0}
    rescan, safe_parts, raw_parts = [], [], []

    for file_name, d in documents:
        a = d.get("audit_summary") or {}
        for k in audit:
            audit[k] += int(a.get(k, 0))

        rfi_pages = [p for p in d["pages"] if p["page_action"] == "TRIGGER_PAGE_RFI_RESCAN"]
        if rfi_pages:
            rescan.append({
                "document_id": d["document_id"],
                "file_name": file_name,
                "pages": [p["page_number"] for p in rfi_pages],
                "rfi_reasons": sorted({r for p in rfi_pages for r in p.get("rfi_reasons", [])}),
            })

        header = f"--- DOCUMENT: {file_name} ({d['detected_doc_category']}) ---"
        if d.get("safe_downstream_text"):
            safe_parts.append(f"{header}\n{d['safe_downstream_text']}")
        if d.get("full_raw_audit_text"):
            raw_parts.append(f"{header}\n{d['full_raw_audit_text']}")

    docs_only = [d for _, d in documents]
    if failed_docs:
        status = "INCOMPLETE_PROCESSING_ERROR"
    elif not docs_only:
        status = "EMPTY_CASE"
    elif rescan:
        status = "RFI_RESCAN_REQUIRED" if len(rescan) == len(docs_only) else "PARTIAL_RFI_RESCAN"
    elif any(d["document_status"] == "PENDING_HUMAN_REVIEW" for d in docs_only):
        status = "PENDING_HUMAN_REVIEW"
    else:
        status = "AUTO_COMPLETE"

    return {
        "case_id": case_id,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "total_documents": len(docs_only),
        "total_pages_across_case": sum(int(d.get("total_pages", 0)) for d in docs_only),
        "case_status": status,
        "case_audit_summary": {**audit, "documents_needing_rescan_rfi": rescan},
        "documents_failed": failed_docs,
        "combined_safe_downstream_text": "\n\n".join(safe_parts),
        "combined_raw_audit_text": "\n\n".join(raw_parts),
        "documents": docs_only,
    }


def _next_pipeline_stage(bundle):
    """PENDING_PII normally; RFI_RESCAN_REQUIRED if ANY document failed the page-level RFI gate;
    None (= leave the stage alone so the next run retries) if a file errored or the case is empty."""
    if bundle["documents_failed"] or bundle["total_documents"] == 0:
        return None
    if bundle["case_audit_summary"]["documents_needing_rescan_rfi"]:
        return STAGE_RFI_RESCAN
    return STAGE_PENDING_PII


# ---------------------------------------------------------------
# 7. Batch runner
# ---------------------------------------------------------------
def run_supabase_batch_pipeline(case_numbers=None, force_reprocess=False):
    if "process_document_to_json" not in globals():
        raise RuntimeError("process_document_to_json is not defined. Run Cell 5 first.")
    if globals().get("supabase") is None:
        raise RuntimeError("`supabase` client is not defined. Run the Supabase connection cell first.")

    cases = fetch_cases_to_process(case_numbers)
    started = datetime.now(timezone.utc)
    manifest = {
        "run_id": started.strftime("%Y%m%dT%H%M%SZ"),
        "started_at_utc": started.isoformat(timespec="seconds"),
        "finished_at_utc": None,
        "batch_status": "RUNNING",
        "force_reprocess": force_reprocess,
        "total_cases": len(cases),
        "total_files": 0,
        "ocr_run": 0, "cached": 0, "failed": 0, "skipped": 0,
        "total_processing_time_sec": 0.0,
        "cases": [],
    }
    print(f"Found {len(cases)} case(s) waiting for OCR")
    batch_t0 = time.perf_counter()
    size_index = {}                                   # folder -> {object name: size}

    for c_idx, case in enumerate(cases, start=1):
        case_uuid, case_number = case["case_id"], case["case_number"]
        rows = fetch_case_documents(case_uuid)
        manifest["total_files"] += len(rows)
        print(f"\n=== Case {c_idx}/{len(cases)}: {case_number} ({len(rows)} file(s)) ===")

        case_entry = {
            "case_uuid": case_uuid, "case_number": case_number,
            "case_status": None, "next_pipeline_stage": None,
            "total_documents": 0, "total_pages_across_case": 0,
            "audit_summary": None, "snippets_uploaded": 0, "documents": [],
        }
        manifest["cases"].append(case_entry)

        documents, failed_docs, seen_ids = [], [], {}

        for row in rows:
            file_name = row["file_name"]
            doc_id = _doc_id_from_name(_safe_filename(file_name))
            entry = {
                "file_name": file_name, "document_id": doc_id, "status": None,
                "detected_doc_category": None, "document_status": None,
                "total_pages": None, "audit_summary": None,
                "gpu_time_sec": 0.0, "error": None,
            }
            print(f"  - {file_name}")

            if Path(file_name).suffix.lower() not in SUPPORTED_EXTS:
                entry.update(status="SKIPPED_UNSUPPORTED_TYPE",
                             error=f"unsupported extension '{Path(file_name).suffix}'")
                manifest["skipped"] += 1
                failed_docs.append({"file_name": file_name, "status": entry["status"], "error": entry["error"]})
                print(f"    SKIPPED: {entry['error']}")

            elif doc_id in seen_ids:
                # Same stem twice in one case would overwrite snippets and collide in the bundle
                entry.update(status="SKIPPED_DUPLICATE_ID",
                             error=f"document_id '{doc_id}' already used by {seen_ids[doc_id]}; rename this file")
                manifest["skipped"] += 1
                failed_docs.append({"file_name": file_name, "status": entry["status"], "error": entry["error"]})
                print(f"    SKIPPED: {entry['error']}")

            else:
                seen_ids[doc_id] = file_name
                try:
                    out = _process_one_document(row, case_number, size_index, force=force_reprocess)
                    doc = out["doc"]
                    documents.append((file_name, doc))
                    entry.update(
                        status=out["status"],
                        document_id=doc["document_id"],
                        detected_doc_category=doc["detected_doc_category"],
                        document_status=doc["document_status"],
                        total_pages=doc["total_pages"],
                        audit_summary=doc["audit_summary"],
                        gpu_time_sec=round(out["gpu_time"], 2),
                    )
                    case_entry["snippets_uploaded"] += out["snippets_uploaded"]
                    manifest["cached" if out["status"] == "CACHED" else "ocr_run"] += 1
                    if out["status"] == "CACHED":
                        print("    CACHED: raw_ocr_json reused (size unchanged); GPU OCR skipped")
                except Exception as e:        # one bad file must not stop the case or the batch
                    entry.update(status="FAILED", error=_clean_error(e))
                    manifest["failed"] += 1
                    failed_docs.append({"file_name": file_name, "status": "FAILED", "error": entry["error"]})
                    print(f"    FAILED: {entry['error']}")
                    try:
                        supabase.table("claim_documents").update(
                            {"processing_status": DOC_FAILED, "rfi_reason": entry["error"]}
                        ).eq("doc_id", row["doc_id"]).execute()
                    except Exception as db_err:
                        print(f"    !! Could not record FAILED status either: {_clean_error(db_err)}")

            case_entry["documents"].append(entry)
            manifest["total_processing_time_sec"] = round(time.perf_counter() - batch_t0, 2)

        # ---- master bundle -> claims_cases ----
        bundle = _json_safe(_build_case_bundle(case_number, documents, failed_docs))
        next_stage = _next_pipeline_stage(bundle)
        case_update = {
            "master_ocr_bundle_json": bundle,
            "case_audit_summary": bundle["case_audit_summary"],
        }
        if next_stage:
            case_update["pipeline_stage"] = next_stage
        supabase.table("claims_cases").update(case_update).eq("case_id", case_uuid).execute()

        case_entry.update(
            case_status=bundle["case_status"],
            next_pipeline_stage=next_stage or f"{case['pipeline_stage']} (unchanged)",
            total_documents=bundle["total_documents"],
            total_pages_across_case=bundle["total_pages_across_case"],
            audit_summary=bundle["case_audit_summary"],
        )
        print(f"  -> {bundle['case_status']}  |  pipeline_stage = {case_entry['next_pipeline_stage']}")

    manifest["finished_at_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    manifest["total_processing_time_sec"] = round(time.perf_counter() - batch_t0, 2)
    manifest["batch_status"] = (
        "COMPLETE" if not manifest["failed"] and not manifest["skipped"] else "COMPLETE_WITH_ISSUES"
    )
    return manifest


# ---------------------------------------------------------------
# 8. Batch telemetry table
# ---------------------------------------------------------------
def _print_table(headers, rows):
    widths = [max(len(h), *(len(r[i]) for r in rows)) if rows else len(h) for i, h in enumerate(headers)]
    sep = "-+-".join("-" * w for w in widths)
    print(" | ".join(h.ljust(w) for h, w in zip(headers, widths)))
    print(sep)
    for r in rows:
        print(" | ".join(v.ljust(w) for v, w in zip(r, widths)))
    return len(sep)


def print_batch_summary(manifest):
    rows = []
    for c in manifest["cases"]:
        for d in c["documents"]:
            a = d["audit_summary"] or {}
            rows.append((
                c["case_number"], d["file_name"], d["detected_doc_category"] or "-", d["status"] or "-",
                d["document_status"] or "-", str(d["total_pages"] or "-"),
                f"{a.get('AUTO', 0)} / {a.get('SUGGEST_CONFIRM', 0)} / {a.get('HUMAN_REVIEW', 0)}" if a else "-",
                f"{d['gpu_time_sec']:.1f}s",
            ))
    headers = ("Patient Case ID", "Uploaded Document File", "Detected Category", "OCR Run",
               "Final Document Status", "Pages", "A / C / R Blocks", "GPU Time (s)")

    print(f"\nBATCH {manifest['run_id']}: {manifest['total_cases']} case(s), {manifest['total_files']} file(s) | "
          f"OCR run {manifest['ocr_run']}, cached {manifest['cached']}, "
          f"failed {manifest['failed']}, skipped {manifest['skipped']} | "
          f"{manifest['total_processing_time_sec']}s")
    width = _print_table(headers, rows)
    print("=" * width)

    # ---- per-case summary ----
    case_rows = []
    for c in manifest["cases"]:
        a = c["audit_summary"] or {}
        case_rows.append((
            c["case_number"], c["case_status"] or "-", c["next_pipeline_stage"] or "-",
            str(c["total_documents"]), str(c["total_pages_across_case"]),
            f"{a.get('AUTO', 0)} / {a.get('SUGGEST_CONFIRM', 0)} / {a.get('HUMAN_REVIEW', 0)}",
            str(len(a.get("documents_needing_rescan_rfi", []))), str(c["snippets_uploaded"]),
        ))
    print("\nPer-case summary (claims_cases updated)")
    _print_table(("Patient Case ID", "Case Status", "Pipeline Stage", "Docs", "Pages",
                  "A / C / R Blocks", "Docs Needing Rescan", "Snippets Uploaded"), case_rows)

    for c in manifest["cases"]:
        for d in c["documents"]:
            if d["error"]:
                print(f"  ! {c['case_number']} / {d['file_name']}: {d['error']}")


if __name__ == "__main__":
    manifest = run_supabase_batch_pipeline()
    print_batch_summary(manifest)